In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Subset
import random
from collections import defaultdict
from collections import Counter
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import pandas as pd

Import all necessary libraries

In [2]:
def set_seed(seed):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

rotate = transforms.Compose([
    transforms.RandomRotation(degrees=30),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

flip = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

crop = transforms.Compose([
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0),
        ratio=(0.9, 1.1)
    ),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

erase = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing()
])

affine = transforms.Compose([
    transforms.RandomAffine(
        degrees=10,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05),
        shear=5
    ),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

blur = transforms.Compose([
    transforms.GaussianBlur(3, sigma=(0.1, 0.5)),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [4]:
CAT_LABEL = 3
DOG_LABEL = 5

# Load dataset
total_trainset = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True, download=True, 
    transform=transform
)
rotate_trainset = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True, download=True, 
    transform=rotate
)
flip_trainset = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True, download=True, 
    transform=flip
)
crop_trainset = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True, download=True, 
    transform=crop
)
erase_trainset = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True, download=True, 
    transform=erase
)
affine_trainset = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True, download=True, 
    transform=affine
)
blur_trainset = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True, download=True, 
    transform=blur
)

In [5]:
def balanced_cat_dog_indices(dataset, total_samples, seed):
    rng = random.Random(seed)
    class_indices = defaultdict(list)

    for i, (_, label) in enumerate(dataset):
        if label in [CAT_LABEL, DOG_LABEL]:
            class_indices[label].append(i)

    samples_per_class = total_samples // 2

    cat_indices = rng.sample(class_indices[CAT_LABEL], samples_per_class)
    dog_indices = rng.sample(class_indices[DOG_LABEL], samples_per_class)

    indices = cat_indices + dog_indices
    rng.shuffle(indices)

    return indices


class CatDogDataset(torch.utils.data.Dataset):
    def __init__(self, subset):
        self.subset = subset

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]
        label = 0 if label == CAT_LABEL else 1
        return image, label


full_testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

test_indices = [
    i for i, (_, label) in enumerate(full_testset)
    if label in [CAT_LABEL, DOG_LABEL]
]

testset = CatDogDataset(Subset(full_testset, test_indices))

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=64,
    shuffle=False
)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"


def make_model():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 2)
    return model


def train_model(model, trainloader, epochs=5, lr=0.001):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    final_train_loss = None

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for images, labels in trainloader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        final_train_loss = running_loss / len(trainloader)

    return model, final_train_loss


def evaluate_model(model, testloader):
    model.eval()

    criterion = nn.CrossEntropyLoss()
    correct = 0
    total = 0
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in testloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_accuracy = correct / total
    test_loss = running_loss / len(testloader)

    return test_accuracy, test_loss

In [7]:
def extract_features(model, dataloader):
    model.eval()

    feature_extractor = nn.Sequential(*list(model.children())[:-1])
    feature_extractor = feature_extractor.to(device)

    features = []
    labels_list = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)

            batch_features = feature_extractor(images)
            batch_features = batch_features.view(batch_features.size(0), -1)

            features.append(batch_features.cpu().numpy())
            labels_list.extend(labels.numpy())

    features = np.vstack(features)
    labels_list = np.array(labels_list)

    return features, labels_list


def compute_silhouette(features, labels):
    return silhouette_score(features, labels)


def intraclass_dispersion(features, labels):
    dispersions = []

    for class_label in np.unique(labels):
        class_features = features[labels == class_label]
        class_center = class_features.mean(axis=0)

        distances = np.linalg.norm(class_features - class_center, axis=1)
        dispersions.append(distances.mean())

    return np.mean(dispersions)

In [8]:
def run_one_condition(
    condition_name,
    train_dataset,
    train_indices,
    batch_size,
    seed,
    subset_id,
    epochs=5,
    return_model=False
):
    set_seed(seed)

    train_subset = Subset(train_dataset, train_indices)
    train_catdog = CatDogDataset(train_subset)

    generator = torch.Generator()
    generator.manual_seed(seed)

    trainloader = torch.utils.data.DataLoader(
        train_catdog,
        batch_size=batch_size,
        shuffle=True,
        generator=generator
    )

    model = make_model()

    print(f"\nStarting: {condition_name} | seed={seed} | subset={subset_id}")

    model, train_loss = train_model(model, trainloader, epochs=epochs)

    print(f"Finished training: {condition_name} | seed={seed} | subset={subset_id}")
    test_accuracy, test_loss = evaluate_model(model, testloader)

    features, labels = extract_features(model, testloader)

    sil_score = compute_silhouette(features, labels)
    dispersion = intraclass_dispersion(features, labels)
    

    result = {
        "condition": condition_name,
        "seed": seed,
        "subset_id": subset_id,
        "train_loss": train_loss,
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
        "silhouette_score": sil_score,
        "intraclass_dispersion": dispersion
    }

    if return_model:
        return result, model, features, labels

    return result

In [ ]:
seeds = [1, 2, 3, 4, 5]
subset_ids = [1]

results = []

for seed in seeds:
    for subset_id in subset_ids:
        subset_seed = seed * 1000 + subset_id

        indices_500 = balanced_cat_dog_indices(
            total_trainset,
            total_samples=500,
            seed=subset_seed
        )

        # 500-base
        results.append(
            run_one_condition(
                condition_name="500-base",
                train_dataset=total_trainset,
                train_indices=indices_500,
                batch_size=40,
                seed=seed,
                subset_id=subset_id,
                epochs=5
            )
        )

        # 500-rotate
        results.append(
            run_one_condition(
                condition_name="500-rotate",
                train_dataset=rotate_trainset,
                train_indices=indices_500,
                batch_size=40,
                seed=seed,
                subset_id=subset_id,
                epochs=5
            )
        )

        # 500-flip
        results.append(
            run_one_condition(
                condition_name="500-flip",
                train_dataset=flip_trainset,
                train_indices=indices_500,
                batch_size=40,
                seed=seed,
                subset_id=subset_id,
                epochs=5
            )
        )

        # 500-crop
        results.append(
            run_one_condition(
                condition_name="500-crop",
                train_dataset=crop_trainset,
                train_indices=indices_500,
                batch_size=40,
                seed=seed,
                subset_id=subset_id,
                epochs=5
            )
        )

        # 500-erase
        results.append(
            run_one_condition(
                condition_name="500-erase",
                train_dataset=erase_trainset,
                train_indices=indices_500,
                batch_size=40,
                seed=seed,
                subset_id=subset_id,
                epochs=5
            )
        )

        # 500-affine
        results.append(
            run_one_condition(
                condition_name="500-affine",
                train_dataset=affine_trainset,
                train_indices=indices_500,
                batch_size=40,
                seed=seed,
                subset_id=subset_id,
                epochs=5
            )
        )

        # 500-blur
        results.append(
            run_one_condition(
                condition_name="500-blur",
                train_dataset=blur_trainset,
                train_indices=indices_500,
                batch_size=40,
                seed=seed,
                subset_id=subset_id,
                epochs=5
            )
        )

        # 5000-image conditions
        indices_5000 = balanced_cat_dog_indices(
            total_trainset,
            total_samples=5000,
            seed=seed
        )

        results.append(
            run_one_condition(
                condition_name="5000-base",
                train_dataset=total_trainset,
                train_indices=indices_5000,
                batch_size=160,
                seed=seed,
                subset_id=subset_id,
                epochs=10
            )
        )

        results.append(
            run_one_condition(
                condition_name="5000-rotate",
                train_dataset=rotate_trainset,
                train_indices=indices_5000,
                batch_size=160,
                seed=seed,
                subset_id=subset_id,
                epochs=10
            )
        )

        results.append(
            run_one_condition(
                condition_name="5000-flip",
                train_dataset=flip_trainset,
                train_indices=indices_5000,
                batch_size=160,
                seed=seed,
                subset_id=subset_id,
                epochs=10
            )
        )

        results.append(
            run_one_condition(
                condition_name="5000-crop",
                train_dataset=crop_trainset,
                train_indices=indices_5000,
                batch_size=160,
                seed=seed,
                subset_id=subset_id,
                epochs=10
            )
        )

        results.append(
            run_one_condition(
                condition_name="5000-erase",
                train_dataset=erase_trainset,
                train_indices=indices_5000,
                batch_size=160,
                seed=seed,
                subset_id=subset_id,
                epochs=10
            )
        )

        results.append(
            run_one_condition(
                condition_name="5000-affine",
                train_dataset=affine_trainset,
                train_indices=indices_5000,
                batch_size=160,
                seed=seed,
                subset_id=subset_id,
                epochs=10
            )
        )

        results.append(
            run_one_condition(
                condition_name="5000-blur",
                train_dataset=blur_trainset,
                train_indices=indices_5000,
                batch_size=160,
                seed=seed,
                subset_id=subset_id,
                epochs=10
            )
        )

results_df = pd.DataFrame(results)
results_df

summary = results_df.groupby("condition").agg(
    test_accuracy_mean=("test_accuracy", "mean"),
    test_accuracy_sd=("test_accuracy", "std"),

    test_loss_mean=("test_loss", "mean"),
    test_loss_sd=("test_loss", "std"),

    train_loss_mean=("train_loss", "mean"),
    train_loss_sd=("train_loss", "std"),

    silhouette_mean=("silhouette_score", "mean"),
    silhouette_sd=("silhouette_score", "std"),

    intraclass_dispersion_mean=("intraclass_dispersion", "mean"),
    intraclass_dispersion_sd=("intraclass_dispersion", "std"),

    n_runs=("condition", "count")
).reset_index()

summary 


Starting: 500-base | seed=1 | subset=1
Finished training: 500-base | seed=1 | subset=1

Starting: 500-rotate | seed=1 | subset=1
Finished training: 500-rotate | seed=1 | subset=1

Starting: 500-flip | seed=1 | subset=1
Finished training: 500-flip | seed=1 | subset=1

Starting: 500-crop | seed=1 | subset=1
Finished training: 500-crop | seed=1 | subset=1

Starting: 500-erase | seed=1 | subset=1
Finished training: 500-erase | seed=1 | subset=1

Starting: 500-affine | seed=1 | subset=1
Finished training: 500-affine | seed=1 | subset=1

Starting: 500-blur | seed=1 | subset=1
Finished training: 500-blur | seed=1 | subset=1

Starting: 5000-base | seed=1 | subset=1
Finished training: 5000-base | seed=1 | subset=1

Starting: 5000-rotate | seed=1 | subset=1


In [ ]:
def plot_pca_from_features(features, labels, title):
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(features)

    plt.figure(figsize=(8, 6))
    for class_label, class_name, color in [(0, "Cat", "blue"), (1, "Dog", "red")]:
        mask = labels == class_label
        plt.scatter(
            reduced[mask, 0],
            reduced[mask, 1],
            label=class_name,
            alpha=0.6,
            s=20
        )

    plt.title(title)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.legend()
    plt.grid(True)
    plt.show()

    
representative_subset_seed = 1001
representative_indices_500 = balanced_cat_dog_indices(
    total_trainset,
    total_samples=500,
    seed=representative_subset_seed
)

pca_conditions = [
    ("500-base", total_trainset, 40),
    ("500-rotate", rotate_trainset, 40),
    ("500-flip", flip_trainset, 40),
    ("500-crop", crop_trainset, 40),
    ("500-erase", erase_trainset, 40),
    ("500-affine", affine_trainset, 40),
    ("500-blur", blur_trainset, 40),
]

for condition_name, dataset_obj, batch_size in pca_conditions:
    result, model, features, labels = run_one_condition(
        condition_name=condition_name,
        train_dataset=dataset_obj,
        train_indices=representative_indices_500,
        batch_size=batch_size,
        seed=1,
        subset_id="representative",
        epochs=5,
        return_model=True
    )

    plot_pca_from_features(
        features,
        labels,
        title=f"{condition_name} PCA (representative run)"
    )

Train the final layers of each!